# Experiment 22: Stacked Calibration Layer on Hybrid ELO+Pi Poisson GLM

**Goal:** add a second-stage calibration/stacking layer on top of Experiment D (hybrid ELO+Pi Poisson GLM) and evaluate with the same 4-fold LOTO-CV setup.

Pipeline:
1. Fit base hybrid GLM per fold and generate base lambdas/predictions.
2. Build a meta-dataset from base predictions + contextual features.
3. For each target fold, train Poisson meta-models on the other 3 folds only (cross-fold stacking).
4. Compare stacked vs base using mean Sporza points and WC-2022 fold delta.

In [ ]:
import sys
sys.path.insert(0, '../../src')

import json
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from penaltyblog.ratings import PiRatingSystem

from evaluation.sporza import sporza_points_series

DATA = Path('../../data/processed')
INTERIM = Path('../../data/interim')
WC_YEARS = [2010, 2014, 2018, 2022]
MAX_GOALS = 8

WC_CUTOFFS = {
    2010: '2010-06-01',
    2014: '2014-06-01',
    2018: '2018-06-01',
    2022: '2022-11-01',
}

HOME_FEATURES = [
    'home_elo', 'away_elo', 'elo_diff',
    'pi_home', 'pi_away', 'pi_diff',
    'is_neutral',
    'home_form_wr', 'away_form_wr',
    'home_form_gf', 'away_form_gf',
    'home_form_ga', 'away_form_ga',
    'tournament_weight',
]
AWAY_FEATURES = [
    'away_elo', 'home_elo', 'elo_diff',
    'pi_away', 'pi_home', 'pi_diff',
    'is_neutral',
    'away_form_wr', 'home_form_wr',
    'away_form_gf', 'home_form_gf',
    'away_form_ga', 'home_form_ga',
    'tournament_weight',
]

META_FEATURES = [
    'base_lh', 'base_la', 'elo_diff', 'pi_diff', 'is_neutral', 'tournament_weight'
]

manifest = json.loads((DATA / 'split_manifest.json').read_text())
autofill_mean = manifest['autofill_baseline']['pooled_mean_pts']
print(f'Autofill baseline: {autofill_mean:.3f} pts/match')

In [ ]:
def build_X(df: pd.DataFrame, features: list[str]) -> np.ndarray:
    X = df[features].copy()
    return X.fillna(X.median(numeric_only=True)).values


def build_pipe(alpha: float = 0.1) -> Pipeline:
    return Pipeline([
        ('scaler', StandardScaler()),
        ('poisson', PoissonRegressor(alpha=alpha, max_iter=400)),
    ])


def bootstrap_ci(pts: pd.Series, n_boot: int = 10000) -> dict:
    rng = np.random.default_rng(42)
    vals = pts.values
    boot = np.array([rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {'mean': float(vals.mean()), 'ci_lo': float(lo), 'ci_hi': float(hi)}


def permutation_test(pts_a: np.ndarray, pts_b: np.ndarray, n_perm: int = 10000) -> float:
    rng = np.random.default_rng(42)
    obs_diff = float(np.mean(pts_a) - np.mean(pts_b))
    diffs = np.zeros(n_perm)
    for i in range(n_perm):
        swap = rng.integers(0, 2, size=len(pts_a)).astype(bool)
        a = np.where(swap, pts_b, pts_a)
        b = np.where(swap, pts_a, pts_b)
        diffs[i] = np.mean(a) - np.mean(b)
    return float(np.mean(diffs >= obs_diff))


@lru_cache(maxsize=60000)
def best_score_cached(lh_r: float, la_r: float) -> tuple[int, int]:
    goals = np.arange(MAX_GOALS + 1)
    ph_pmf = poisson.pmf(goals, lh_r)
    pa_pmf = poisson.pmf(goals, la_r)
    joint = np.outer(ph_pmf, pa_pmf)

    best_pts, best_h, best_a = -1.0, 1, 0
    for ph in range(MAX_GOALS + 1):
        for pa in range(MAX_GOALS + 1):
            exp_pts = 0.0
            for ah in range(MAX_GOALS + 1):
                for aa in range(MAX_GOALS + 1):
                    p = joint[ah, aa]
                    if p < 1e-10:
                        continue
                    if ph == ah and pa == aa:
                        exp_pts += p * 10
                    elif (ph - pa) == (ah - aa):
                        exp_pts += p * 7
                    elif np.sign(ph - pa) == np.sign(ah - aa):
                        exp_pts += p * 5
                    else:
                        exp_pts += p
            if exp_pts > best_pts:
                best_pts, best_h, best_a = exp_pts, ph, pa
    return best_h, best_a


def compute_pi_ratings_before_cutoff(matches: pd.DataFrame, cutoff_date: str) -> dict:
    pi = PiRatingSystem(alpha=0.15, beta=0.10, k=0.75)
    training_matches = matches[matches['date'] < cutoff_date].sort_values('date')
    for _, row in training_matches.iterrows():
        goal_diff = int(row['home_score'] - row['away_score'])
        pi.update_ratings(row['home_team'], row['away_team'], goal_diff)
    return pi.team_ratings


def add_pi_features(df: pd.DataFrame, team_ratings: dict) -> pd.DataFrame:
    out = df.copy()
    out['pi_home'] = out['home_team'].map(lambda t: team_ratings[t]['home'] if t in team_ratings else 0.0)
    out['pi_away'] = out['away_team'].map(lambda t: team_ratings[t]['away'] if t in team_ratings else 0.0)
    out['pi_diff'] = out['pi_home'] - out['pi_away']
    return out

In [ ]:
# Phase 1: base hybrid model predictions per fold (Experiment D replication)
matches_full = pd.read_parquet(INTERIM / 'matches.parquet')

pi_ratings_by_year = {
    year: compute_pi_ratings_before_cutoff(matches_full, cutoff)
    for year, cutoff in WC_CUTOFFS.items()
}

base_fold_rows = []
base_fold_summary = []

for year in WC_YEARS:
    train = pd.read_parquet(DATA / f'train_fold_{year}.parquet')
    evalf = pd.read_parquet(DATA / f'eval_fold_{year}.parquet')

    train = add_pi_features(train, pi_ratings_by_year[year])
    evalf = add_pi_features(evalf, pi_ratings_by_year[year])

    X_tr = np.vstack([build_X(train, HOME_FEATURES), build_X(train, AWAY_FEATURES)])
    y_tr = np.concatenate([train['home_score'].values, train['away_score'].values])

    base_pipe = build_pipe(alpha=0.1)
    base_pipe.fit(X_tr, y_tr)

    lh = np.clip(base_pipe.predict(build_X(evalf, HOME_FEATURES)), 0.01, None)
    la = np.clip(base_pipe.predict(build_X(evalf, AWAY_FEATURES)), 0.01, None)

    base_scores = [best_score_cached(round(float(h), 2), round(float(a), 2)) for h, a in zip(lh, la)]
    base_pred_home = pd.Series([s[0] for s in base_scores], index=evalf.index)
    base_pred_away = pd.Series([s[1] for s in base_scores], index=evalf.index)
    base_pts = sporza_points_series(base_pred_home.reset_index(drop=True), base_pred_away.reset_index(drop=True), evalf['home_score'].reset_index(drop=True), evalf['away_score'].reset_index(drop=True))

    fold_meta = evalf[['home_score', 'away_score', 'elo_diff', 'pi_diff', 'is_neutral', 'tournament_weight']].copy()
    fold_meta['year'] = year
    fold_meta['base_lh'] = lh
    fold_meta['base_la'] = la
    fold_meta['base_pred_home'] = base_pred_home.values
    fold_meta['base_pred_away'] = base_pred_away.values
    fold_meta['base_pts'] = base_pts.values
    base_fold_rows.append(fold_meta)

    base_fold_summary.append({'year': year, 'base_mean_pts': float(base_pts.mean())})

meta_df = pd.concat(base_fold_rows, ignore_index=True)
base_ci = bootstrap_ci(meta_df['base_pts'])

print('Base hybrid fold means:')
print(pd.DataFrame(base_fold_summary).to_string(index=False))
print(f"\nBase pooled: {base_ci['mean']:.3f} pts/match  95% CI [{base_ci['ci_lo']:.3f}, {base_ci['ci_hi']:.3f}]")

In [ ]:
# Phase 2: cross-fold stacked calibration layer
stacked_eval_rows = []
stacked_fold_summary = []

for year in WC_YEARS:
    meta_train = meta_df[meta_df['year'] != year].copy()
    meta_eval = meta_df[meta_df['year'] == year].copy()

    X_meta_train = build_X(meta_train, META_FEATURES)
    X_meta_eval = build_X(meta_eval, META_FEATURES)

    home_cal = build_pipe(alpha=0.2)
    away_cal = build_pipe(alpha=0.2)
    home_cal.fit(X_meta_train, meta_train['home_score'].values)
    away_cal.fit(X_meta_train, meta_train['away_score'].values)

    stack_lh = np.clip(home_cal.predict(X_meta_eval), 0.01, None)
    stack_la = np.clip(away_cal.predict(X_meta_eval), 0.01, None)

    stack_scores = [best_score_cached(round(float(h), 2), round(float(a), 2)) for h, a in zip(stack_lh, stack_la)]
    stack_pred_home = pd.Series([s[0] for s in stack_scores], index=meta_eval.index)
    stack_pred_away = pd.Series([s[1] for s in stack_scores], index=meta_eval.index)

    stack_pts = sporza_points_series(stack_pred_home.reset_index(drop=True), stack_pred_away.reset_index(drop=True), meta_eval['home_score'].reset_index(drop=True), meta_eval['away_score'].reset_index(drop=True))

    out = meta_eval.copy()
    out['stack_lh'] = stack_lh
    out['stack_la'] = stack_la
    out['stack_pred_home'] = stack_pred_home.values
    out['stack_pred_away'] = stack_pred_away.values
    out['stack_pts'] = stack_pts.values
    stacked_eval_rows.append(out)

    stacked_fold_summary.append({
        'year': year,
        'base_mean_pts': float(meta_eval['base_pts'].mean()),
        'stack_mean_pts': float(stack_pts.mean()),
        'delta_stack_minus_base': float(stack_pts.mean() - meta_eval['base_pts'].mean()),
    })

stacked_eval = pd.concat(stacked_eval_rows, ignore_index=True)
stack_ci = bootstrap_ci(stacked_eval['stack_pts'])

base_pts_arr = stacked_eval['base_pts'].to_numpy()
stack_pts_arr = stacked_eval['stack_pts'].to_numpy()
p_stack_vs_base = permutation_test(stack_pts_arr, base_pts_arr)

summary_df = pd.DataFrame(stacked_fold_summary)
print(summary_df.to_string(index=False))

print(f"\nStacked pooled: {stack_ci['mean']:.3f} pts/match  95% CI [{stack_ci['ci_lo']:.3f}, {stack_ci['ci_hi']:.3f}]")
print(f"Base pooled:    {base_ci['mean']:.3f} pts/match  95% CI [{base_ci['ci_lo']:.3f}, {base_ci['ci_hi']:.3f}]")
print(f"Delta vs base:  {stack_ci['mean'] - base_ci['mean']:+.3f}")
print(f"Delta vs autofill ({autofill_mean:.3f}): {stack_ci['mean'] - autofill_mean:+.3f}")
print(f"Permutation test (stacked > base): p = {p_stack_vs_base:.4f}")

wc2022_delta = summary_df.loc[summary_df['year'] == 2022, 'delta_stack_minus_base'].iloc[0]
print(f"WC 2022 fold delta (stacked - base): {wc2022_delta:+.3f}")